# 🏥 Projeto Ciência de Dados — MPTI/IFPB 2026.1

**Disciplina:** Ciência de Dados — MPTI/IFPB — 2026.1  
**Docentes:** Damires Souza e Alex Cunha  
**Equipe:** Rodrigo de Queiroz Gonçalves Velez · Caio Jordan de Lima Maia

---

## Contexto

As Doenças Crônicas Não Transmissíveis (DCNT) — cardiovasculares, diabetes, câncer e doenças respiratórias crônicas — são responsáveis por uma parcela significativa da mortalidade no Brasil. A análise desses dados em nível municipal é fundamental para apoiar políticas públicas de saúde, permitindo identificar desigualdades regionais e orientar a distribuição de recursos.

Este projeto integra bases de dados públicas (DataSUS e IBGE) para identificar padrões e perfis municipais de mortalidade por DCNT, aplicando técnicas de ciência de dados: limpeza e integração de dados heterogêneos, análise exploratória, clustering e visualização de resultados.

## Problema de Ciência de Dados

Identificar padrões e perfis de municípios brasileiros em relação à mortalidade por DCNT, a partir da integração de diferentes bases de dados públicas. Especificamente:

- Analisar a distribuição da mortalidade por DCNT em nível municipal
- Identificar agrupamentos de municípios com características semelhantes (clustering)
- Explorar relações entre indicadores de mortalidade e variáveis demográficas

## Fontes de Dados

| Fonte | Descrição | Acesso | Formato bruto |
|-------|-----------|--------|---------------|
| **DataSUS / SIM** | Sistema de Informações sobre Mortalidade — declarações de óbito com causas por CID-10, por município, estado e ano | FTP ftp.datasus.gov.br | DBC |
| **DataSUS / CID-10 v2008** | Tabelas de referência CID-10: capítulos, grupos, categorias, subcategorias e CID-O | HTTP www2.datasus.gov.br | ZIP/CSV |
| **DataSUS / CID-10 FTP** | Tabelas de referência CID-10 em DBF: categorias+subcategorias e capítulos | FTP ftp.datasus.gov.br | DBF |
| **IBGE / POPSVS** | Projeções populacionais por município, disponibilizadas via FTP do DataSUS | FTP ftp.datasus.gov.br | ZIP/DBF |
| **IBGE API REST** | Cadastro oficial de municípios com código IBGE, microrregião, mesorregião e UF | API servicodados.ibge.gov.br | JSON |

Todos os dados são secundários, já agregados e anonimizados pelos órgãos responsáveis — o projeto está em conformidade com a LGPD.

---

## Arquitetura Medallion

Este notebook cobre a **camada Bronze**: coleta e conversão dos dados brutos das fontes originais para CSV, sem qualquer transformação de conteúdo.

```
🥉 Bronze  →  🥈 Prata  →  🥇 Ouro
(coleta)      (limpeza)    (análise)
```

**Período coberto:** 2010–2024 (15 exercícios)


## ⚙️ Célula 0 — Inicialização: dependências, Google Drive e constantes

Instala as dependências Python necessárias (`pyreaddbc`, `dbfread`, `pandas`, `pyarrow`), define `DATA_DIR` como a pasta `dados/` local (relativa ao notebook) e todas as constantes do projeto: caminhos de pasta, endereços FTP/HTTP, lista de anos (2015–2024) e siglas dos 27 estados. Os diretórios da arquitetura Medallion são criados automaticamente se ainda não existirem.


In [156]:
# ── Instalação de dependências ────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyreaddbc', 'dbfread', 'pandas', 'pyarrow'], check=True)
from pathlib import Path

# ── Pasta de dados (relativa ao notebook) ────────────────────────────────────
# Em VSCode/Jupyter local aponta para a pasta dados/ no raiz do projeto.
# Em Colab online ajuste para um caminho absoluto, ex.: Path('/content/dados')
DATA_DIR = Path('dados')

# ── Constantes FTP / URL ─────────────────────────────────────────────────────
FTP_HOST          = 'ftp.datasus.gov.br'
FTP_PATH_SIM      = '/dissemin/publicos/SIM/CID10/DORES/'
FTP_PATH_IBGE_POP = '/dissemin/publicos/IBGE/POPSVS/'
FTP_PATH_CID      = '/dissemin/publicos/SIM/CID10/TABELAS/'
API_IBGE_MUN      = 'https://servicodados.ibge.gov.br/api/v1/localidades/municipios?orderBy=nome'
URL_CID10_V2008   = 'http://www2.datasus.gov.br/cid10/V2008/downloads/CID10CSV.zip'

# ── Diretórios por camada Medallion ─────────────────────────────────────────
BRONZE_DIR     = DATA_DIR / '1-bronze'
PRATA_DIR      = DATA_DIR / '2-prata'
OURO_DIR       = DATA_DIR / '3-ouro'

CID_V2008_BASE = BRONZE_DIR / 'cid_10_datasus_v2008'
CID_FTP_BASE   = BRONZE_DIR / 'cid_10_datasus_ftp'
SIM_BASE       = BRONZE_DIR / 'SIM'
IBGE_MUN_BASE  = BRONZE_DIR / 'ibge_dados_municipios'
IBGE_POP_BASE  = BRONZE_DIR / 'ibge_populacao'

# ── Parâmetros de coleta ─────────────────────────────────────────────────────
ANOS = list(range(2010, 2025))

ESTADOS = [
    'AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO',
    'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR',
    'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO',
]
CID_FTP_TABELAS = ['CID10.DBF', 'CIDCAP10.DBF']

# ── Cria estrutura de pastas (sem efeito se já existir) ──────────────────────
for _p in ['1-bronze', '2-prata', '3-ouro']:
    (DATA_DIR / _p).mkdir(parents=True, exist_ok=True)

print(f'Pasta de dados       : {DATA_DIR.resolve()}')
print(f'Dados → Bronze.......: {BRONZE_DIR}')
print(f'Dados → Prata........: {PRATA_DIR}')
print(f'Dados → Ouro.........: {OURO_DIR}')
print(f'\nAnos configurados    : {ANOS[0]}\u2013{ANOS[-1]}  ({len(ANOS)} exercícios)')
print(f'Estados configurados : {len(ESTADOS)} UFs')
print('\n\u2705 Inicialização concluída.')


Pasta de dados       : /Users/rodrigo/Dados/git/mpti-cd_202601-projeto/dados
Dados → Bronze.......: dados/1-bronze
Dados → Prata........: dados/2-prata
Dados → Ouro.........: dados/3-ouro

Anos configurados    : 2010–2024  (15 exercícios)
Estados configurados : 27 UFs

✅ Inicialização concluída.


## 🔧 Célula 1 — Imports e utilitários compartilhados

Importa todas as bibliotecas usadas no pipeline e define três funções reutilizáveis:

- **`conectar_ftp()`** — Abre conexão FTP anônima com o DataSUS em modo passivo.
- **`dbc_para_parquet(dbc, parquet)`** — Descomprime o DBC para um DBF temporário via `pyreaddbc`, lê com `dbfread`, salva como Parquet e apaga automaticamente o DBC de origem após sucesso.
- **`dbf_para_parquet(dbf, parquet)`** — Converte DBF diretamente para CSV e apaga o DBF de origem após sucesso.
- **`_limpar_dir_vazio(path)`** — Remove diretórios que ficaram vazios após a exclusão dos arquivos intermediários.


In [157]:
# ── Imports e utilitários compartilhados ──────────────────────────────────────
import ftplib, ssl, gzip, io, re, sys, zipfile, tempfile
import json as _json
import shutil
from pathlib import Path
from urllib.request import urlopen
from urllib.error import URLError

import pandas as pd
import pyreaddbc.readdbc as _readdbc
from dbfread import DBF

def conectar_ftp():
    """Abre conexão FTP anônima com o DataSUS em modo passivo."""
    print(f'Conectando ao FTP {FTP_HOST}...')
    ftp = ftplib.FTP(FTP_HOST, timeout=60)
    ftp.login()
    ftp.set_pasv(True)
    print('Conectado.\n')
    return ftp

def _limpar_dir_vazio(path: Path, niveis: int = 2) -> None:
    """Remove diretórios vazios, subindo até `niveis` níveis acima."""
    for _ in range(niveis):
        try:
            if path.exists() and not any(path.iterdir()):
                path.rmdir()
                path = path.parent
            else:
                break
        except OSError:
            break

def dbc_para_parquet(dbc: Path, parquet: Path) -> bool:
    """
    Converte DBC para Parquet via pyreaddbc + dbfread.
    Apaga o DBC de origem após conversão bem-sucedida (ou se Parquet já existia).
    """
    sucesso = False
    tmp_dbf = None
    if parquet.exists():
        print(f'  [SKIP] {parquet.name}')
        sucesso = True
    else:
        parquet.parent.mkdir(parents=True, exist_ok=True)
        try:
            with tempfile.NamedTemporaryFile(suffix='.dbf', delete=False) as tmp:
                tmp_dbf = Path(tmp.name)
            _readdbc.dbc2dbf(str(dbc), str(tmp_dbf))
            df = pd.DataFrame(iter(DBF(str(tmp_dbf), encoding='latin-1')))
            df.to_parquet(parquet, index=False)
            print(f'  [OK]   {parquet.name}  ({len(df):,} linhas, {len(df.columns)} colunas)')
            sucesso = True
        except Exception as e:
            print(f'  [ERR]  {dbc.name}: {e}')
            parquet.unlink(missing_ok=True)
        finally:
            if tmp_dbf and tmp_dbf.exists():
                tmp_dbf.unlink()

    return sucesso

def dbc_para_csv(dbc: Path, csv: Path) -> bool:
    """
    Converte DBC para Parquet via pyreaddbc + dbfread.
    Apaga o DBC de origem após conversão bem-sucedida (ou se Parquet já existia).
    """
    sucesso = False
    tmp_dbf = None
    if csv.exists():
        print(f'  [SKIP] {csv.name}')
        sucesso = True
    else:
        csv.parent.mkdir(parents=True, exist_ok=True)
        try:
            with tempfile.NamedTemporaryFile(suffix='.dbf', delete=False) as tmp:
                tmp_dbf = Path(tmp.name)
            _readdbc.dbc2dbf(str(dbc), str(tmp_dbf))
            df = pd.DataFrame(iter(DBF(str(tmp_dbf), encoding='latin-1')))
            df.to_csv(csv, index=False)
            print(f'  [OK]   {csv.name}  ({len(df):,} linhas, {len(df.columns)} colunas)')
            sucesso = True
        except Exception as e:
            print(f'  [ERR]  {dbc.name}: {e}')
            csv.unlink(missing_ok=True)
        finally:
            if tmp_dbf and tmp_dbf.exists():
                tmp_dbf.unlink()

    return sucesso

def dbf_para_csv(dbf: Path, csv: Path) -> bool:
    """
    Converte DBF para CSV via dbfread.
    """
    sucesso = False
    if csv.exists():
        print(f'  [SKIP] {csv.name}')
        sucesso = True
    else:
        csv.parent.mkdir(parents=True, exist_ok=True)
        try:
            df = pd.DataFrame(iter(DBF(str(dbf), encoding='latin-1')))
            df.to_csv(csv, index=False)
            print(f'  [OK]   {csv.name}  ({len(df):,} linhas, {len(df.columns)} colunas)')
            sucesso = True
        except Exception as e:
            print(f'  [ERR]  {dbf.name}: {e}')
            csv.unlink(missing_ok=True)

    return sucesso


def dbf_para_parquet(dbf_path: Path, parquet_path: Path) -> bool:
    """
    Converte DBF para Parquet via dbfread.
    Apaga o DBF de origem após conversão bem-sucedida (ou se Parquet já existia).
    """
    sucesso = False
    if parquet_path.exists():
        print(f'  [SKIP] {parquet_path.name}')
        sucesso = True
    else:
        parquet_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            df = pd.DataFrame(iter(DBF(str(dbf_path), encoding='latin-1')))
            df.to_parquet(parquet_path, index=False)
            df.to_csv(parquet_path.with_suffix('.csv'), index=False, sep=';', encoding='utf-8-sig')
            print(f'  [OK]   {parquet_path.name}  ({len(df):,} linhas, {len(df.columns)} colunas)')
            sucesso = True
        except Exception as e:
            print(f'  [ERR]  {dbf_path.name}: {e}')
            parquet_path.unlink(missing_ok=True)

    return sucesso

print('\u2705 Utilitários compartilhados carregados.')

✅ Utilitários compartilhados carregados.


## ⚙️ Célula 2 — Configuração do pipeline

Centraliza as opções ajustáveis sem necessidade de editar o código das etapas. A variável `FONTE_CID10` controla qual fonte de CID-10 será usada: `None` para automático (v2008 com fallback para FTP), `'v2008'` ou `'ftp'` para forçar uma fonte específica.

Fonte CID-10:  
  None    → automático: v2008 com fallback para ftp.  
  'v2008' → somente v2008 (mais completo: capítulos, grupos, categorias).  
  'ftp'   → somente ftp  (sem dados de grupos)


In [158]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURAÇÃO — edite aqui antes de executar o pipeline  ║
# ╚══════════════════════════════════════════════════════════════╝

# Fonte CID-10:
#   None    → automático: v2008 com fallback para ftp
#   'v2008' → somente v2008 (mais completo: capítulos, grupos, categorias)
#   'ftp'   → somente ftp  (sem dados de grupos)
FONTE_CID10 = None

print('⚙️  Configuração ativa:')
print(f'   Dados   : {DATA_DIR}')
print(f'   Anos    : {ANOS[0]}–{ANOS[-1]}  ({len(ANOS)} exercícios)')
print(f'   Estados : {len(ESTADOS)} UFs  ({", ".join(ESTADOS[:5])}...)')
print(f'   CID-10  : {FONTE_CID10 or "automático (v2008 → ftp)"}')


⚙️  Configuração ativa:
   Dados   : dados
   Anos    : 2010–2024  (15 exercícios)
   Estados : 27 UFs  (AC, AL, AM, AP, BA...)
   CID-10  : automático (v2008 → ftp)


---

## 🥉 BRONZE — Coleta e conversão dos dados brutos

Esta seção realiza o download de todas as fontes de dados e converte os formatos proprietários (DBC, DBF, ZIP) para CSV. Nenhum conteúdo é alterado — apenas o formato de armazenamento. Os arquivos intermediários (DBC, DBF, ZIP) são excluídos automaticamente após a geração de cada CSV para economizar espaço no Drive.

Execute as quatro células abaixo em ordem. Downloads completos (2010–2024, 27 UFs) podem levar 15–30 minutos dependendo da conexão.


### Bronze 1/4 — DataSUS SIM (Declarações de Óbito)

Acessa o FTP `ftp.datasus.gov.br` no diretório `/dissemin/publicos/SIM/CID10/DORES/` e baixa um arquivo DBC por estado por ano, no formato `DO{UF}{ANO}.dbc` (ex.: `DOSP2023.dbc`). Cada DBC é então convertido para CSV e o DBC é excluído. Com 27 estados × 15 anos, são esperados **405 arquivos**.

> **Nota:** arquivos de anos anteriores a 2018 podem não existir para todos os estados — o pipeline registra `[--]` nesses casos sem interromper a execução.


In [159]:
# ── Bronze 1/4: DATASUS SIM — Declarações de Óbito ──────────────────────────
# Baixa DBC do FTP e converte para Parquet + CSV.
# Fonte: ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/
print(f'── Bronze 1/4: DATASUS SIM — Declarações de Óbito ──────────────────────────\n')

def _baixar_dbc(ftp, ftp_path, arquivo, destino):
    if destino.exists():
        print(f'  [SKIP] {arquivo}')
        return True
    
    destino.parent.mkdir(parents=True, exist_ok=True)
    try:
        ftp.cwd(ftp_path)
        
        with open(destino, 'wb') as f:
            ftp.retrbinary(f'RETR {arquivo}', f.write)

        print(f'  [OK]   {arquivo}  ({destino.stat().st_size / 1024:.1f} KB)')
        return True
    except ftplib.error_perm:
        print(f'  [--]   {arquivo} não encontrado')
        return False
    except Exception as e:
        print(f'  [ERR]  {arquivo}: {e}')
        destino.unlink(missing_ok=True)
        return False

def bronze_sim(anos=ANOS, estados=ESTADOS):
    nome_arq = lambda uf, ano: f'DO{uf}{ano}.dbc'
    arquivos = [(nome_arq(uf, ano), ano) for ano in anos for uf in estados]

    print(f'\n{"=" * 60}')
    print(f'Bronze — SIM (Declarações de Óbito)')
    print(f'FTP     : {FTP_PATH_SIM}')
    print(f'Anos    : {anos[0]}–{anos[-1]}  ({len(anos)} exercícios)')
    print(f'Estados : {len(estados)} UFs')
    print(f'Arquivos: {len(arquivos)} previstos')
    print(f'{"=" * 60}\n')

    print('[ Etapa 1/3 — Download DBC ]\n')
    # Verifica quais arquivos realmente precisam ser baixados (sem parquet nem dbc local)
    pendentes = [
        (nome, ano) for nome, ano in arquivos
        if not (SIM_BASE / 'dbc' / str(ano) / nome).exists()
    ]

    if not pendentes:
        print(f'  [SKIP] Todos os {len(arquivos)} dbc já existem — FTP dispensado.\n')
    else:
        print(f'  {len(pendentes)} arquivo(s) pendente(s) de download.\n')
        ftp = conectar_ftp()
        for nome, ano in pendentes:
            destino = SIM_BASE / 'dbc' / str(ano) / nome
            _baixar_dbc(ftp, FTP_PATH_SIM, nome, destino)
        try:
            ftp.quit()
        except OSError:
            ftp.close()
        print()

    print('[ Etapa 2/3 — Conversão DBC → Parquet ]\n')
    convertidos_parquet = 0
    for nome, ano in arquivos:
        dbc     = SIM_BASE / 'dbc'     / str(ano) / nome
        parquet = SIM_BASE / 'parquet' / str(ano) / nome.replace('.dbc', '.parquet')

        if not dbc.exists():
            continue
        
        if dbc_para_parquet(dbc, parquet):
            convertidos_parquet += 1

    print('[ Etapa 3/3 — Conversão DBC → CSV ]\n')
    convertidos_csv = 0
    for nome, ano in arquivos:
        dbc = SIM_BASE / 'dbc' / str(ano) / nome
        csv = SIM_BASE / 'csv' / str(ano) / nome.replace('.dbc', '.csv')

        if not dbc.exists():
            continue
        
        if dbc_para_csv(dbc, csv):
            convertidos_csv += 1

    print(f'\nConversão concluída (parquet): {convertidos_parquet} arquivo(s).')
    print(f'Parquets em: {SIM_BASE / "parquet"}/\n\n')

    print(f'\nConversão concluída (csv): {convertidos_csv} arquivo(s).')
    print(f'CSVs em: {SIM_BASE / "csv"}/\n')

bronze_sim(ANOS, ESTADOS)

── Bronze 1/4: DATASUS SIM — Declarações de Óbito ──────────────────────────


Bronze — SIM (Declarações de Óbito)
FTP     : /dissemin/publicos/SIM/CID10/DORES/
Anos    : 2010–2024  (15 exercícios)
Estados : 27 UFs
Arquivos: 405 previstos

[ Etapa 1/3 — Download DBC ]

  [SKIP] Todos os 405 dbc já existem — FTP dispensado.

[ Etapa 2/3 — Conversão DBC → Parquet ]

  [SKIP] DOAC2010.parquet
  [SKIP] DOAL2010.parquet
  [SKIP] DOAM2010.parquet
  [SKIP] DOAP2010.parquet
  [SKIP] DOBA2010.parquet
  [SKIP] DOCE2010.parquet
  [SKIP] DODF2010.parquet
  [SKIP] DOES2010.parquet
  [SKIP] DOGO2010.parquet
  [SKIP] DOMA2010.parquet
  [SKIP] DOMG2010.parquet
  [SKIP] DOMS2010.parquet
  [SKIP] DOMT2010.parquet
  [SKIP] DOPA2010.parquet
  [SKIP] DOPB2010.parquet
  [SKIP] DOPE2010.parquet
  [SKIP] DOPI2010.parquet
  [SKIP] DOPR2010.parquet
  [SKIP] DORJ2010.parquet
  [SKIP] DORN2010.parquet
  [SKIP] DORO2010.parquet
  [SKIP] DORR2010.parquet
  [SKIP] DORS2010.parquet
  [SKIP] DOSC2010.parquet
  [SKIP] 

### Bronze 2/4 — IBGE Municípios (API REST)

Consulta a API REST do IBGE (`servicodados.ibge.gov.br`) para obter o cadastro completo de municípios brasileiros com hierarquia geográfica (microrregião, mesorregião, UF e grande região). O resultado é salvo em JSON (formato original) e em CSV (achatado via `json_normalize`). A busca recupera todos os 5 570 municípios independentemente do filtro de estados.


In [160]:
# ── Bronze 2/4: IBGE Municípios ──────────────────────────────────────────────
# Coleta dados de municípios via API REST do IBGE.
# Sempre busca todos os municípios (referência completa).
# JSON intermediário é preservado no disco.
print(f'── Bronze 2/4: IBGE Municípios ──────────────────────────\n')

def _sigla_uf(m):
    try:
        return m['microrregiao']['mesorregiao']['UF']['sigla']
    except (TypeError, KeyError):
        pass
    try:
        return m['regiao-imediata']['regiao-intermediaria']['UF']['sigla']
    except (TypeError, KeyError):
        return None

def _renomear_colunas(colunas):
    vistas, mapa = set(), {}
    for col in colunas:
        partes = col.split('.')
        novo = (f'{partes[-2]}_{partes[-1]}' if len(partes) > 1 else partes[-1])
        novo = novo.lower().replace('-', '_')
        if novo not in vistas:
            vistas.add(novo)
            mapa[col] = novo
    return mapa

def bronze_municipios(estados_filtro=None):
    ctx = ssl._create_unverified_context()
    sufixo      = ('_' + '_'.join(sorted(estados_filtro))) if estados_filtro else ''
    json_path    = IBGE_MUN_BASE / 'json'    / f'municipios{sufixo}.json'
    parquet_path = IBGE_MUN_BASE / 'parquet' / f'municipios{sufixo}.parquet'

    print(f'\n{"=" * 60}')
    print(f'Bronze — IBGE Municípios')
    print(f'API     : {API_IBGE_MUN}')
    print(f'Filtro  : {(", ".join(estados_filtro) if estados_filtro else "todos os municípios")}')
    print(f'{"=" * 60}\n')

    print('[ Etapa 1/2 — Coleta via API ]\n')
    if parquet_path.exists():
        print(f'  [SKIP] {parquet_path.name} já existe')
    else:
        try:
            with urlopen(API_IBGE_MUN, timeout=60, context=ctx) as resp:
                raw = resp.read()
                if resp.info().get('Content-Encoding') == 'gzip' or raw[:2] == b'\x1f\x8b':
                    raw = gzip.decompress(raw)
                dados = _json.loads(raw.decode('utf-8'))
        except URLError as e:
            print(f'  [ERR]  Falha na requisição: {e}')
            return
        print(f'  [OK]   {len(dados):,} municípios recebidos')
        if estados_filtro:
            dados = [m for m in dados if _sigla_uf(m) in estados_filtro]
            print(f'  [OK]   {len(dados):,} municípios após filtro')

        print('\n[ Etapa 2/2 — Gravação e limpeza ]\n')
        json_path.parent.mkdir(parents=True, exist_ok=True)
        with open(json_path, 'w', encoding='utf-8') as f:
            _json.dump(dados, f, ensure_ascii=False, indent=2)
        print(f'  [OK]   {json_path.name}  ({json_path.stat().st_size / 1024:.1f} KB)')

        parquet_path.parent.mkdir(parents=True, exist_ok=True)
        df = pd.json_normalize(dados, sep='.')

        _fallback = {
            'microrregiao.mesorregiao.UF.id':          'regiao-imediata.regiao-intermediaria.UF.id',
            'microrregiao.mesorregiao.UF.sigla':        'regiao-imediata.regiao-intermediaria.UF.sigla',
            'microrregiao.mesorregiao.UF.nome':         'regiao-imediata.regiao-intermediaria.UF.nome',
            'microrregiao.mesorregiao.UF.regiao.id':    'regiao-imediata.regiao-intermediaria.UF.regiao.id',
            'microrregiao.mesorregiao.UF.regiao.sigla': 'regiao-imediata.regiao-intermediaria.UF.regiao.sigla',
            'microrregiao.mesorregiao.UF.regiao.nome':  'regiao-imediata.regiao-intermediaria.UF.regiao.nome',
        }
        for dest, src in _fallback.items():
            if dest in df.columns and src in df.columns:
                df[dest] = df[dest].fillna(df[src])

        mapa = _renomear_colunas(list(df.columns))
        df = df[list(mapa.keys())].rename(columns=mapa).dropna(axis=1, how='all')
        for col in df.select_dtypes(include='float64').columns:
            df[col] = df[col].astype('Int64')
        df.to_parquet(parquet_path, index=False)
        df.to_csv(parquet_path.with_suffix('.csv'), index=False, sep=';', encoding='utf-8-sig')
        print(f'  [OK]   {parquet_path.name}  ({len(df):,} linhas, {len(df.columns)} colunas)')

    print(f'\nConcluído. Parquet em {parquet_path}\n')

bronze_municipios(estados_filtro=None)

── Bronze 2/4: IBGE Municípios ──────────────────────────


Bronze — IBGE Municípios
API     : https://servicodados.ibge.gov.br/api/v1/localidades/municipios?orderBy=nome
Filtro  : todos os municípios

[ Etapa 1/2 — Coleta via API ]

  [SKIP] municipios.parquet já existe

Concluído. Parquet em dados/1-bronze/ibge_dados_municipios/parquet/municipios.parquet



### Bronze 3/4 — IBGE Projeções Populacionais

Baixa do FTP do DataSUS os arquivos `POPSBR{AA}.zip` (um por ano), extrai o DBF contido em cada ZIP e converte para CSV. Após a geração de cada CSV, o ZIP e o DBF correspondentes são excluídos para liberar espaço. Com 15 anos de dados (2010–2024), são esperados **15 CSVs** com população projetada por município, ano e sexo.


In [161]:
# ── Bronze 3/4: IBGE Projeções Populacionais ─────────────────────────────────
# Baixa ZIP do FTP, extrai DBF, converte para Parquet.
# Arquivos ZIP e DBF são preservados no disco.
# Fonte: ftp://ftp.datasus.gov.br/dissemin/publicos/IBGE/POPSVS/
print(f'── Bronze 3/4: IBGE Projeções Populacionais ──────────────────────────\n')

def _nome_zip_pop(ano):     return f'POPSBR{str(ano)[2:]}.zip'
def _nome_dbf_pop(ano):     return f'POP{str(ano)[2:]}.dbf'
def _nome_csv_pop(ano):     return f'POP{str(ano)[2:]}.csv'
def _nome_parquet_pop(ano): return f'POP{str(ano)[2:]}.parquet'

def _baixar_zip_pop(ftp, arquivo, destino):
    if destino.exists():
        print(f'  [SKIP] {arquivo} (ZIP já existe)')
        return True
    destino.parent.mkdir(parents=True, exist_ok=True)
    try:
        ftp.cwd(FTP_PATH_IBGE_POP)
        with open(destino, 'wb') as f:
            ftp.retrbinary(f'RETR {arquivo}', f.write)
        print(f'  [OK]   {arquivo}  ({destino.stat().st_size / 1024:.1f} KB)')
        return True
    except ftplib.error_perm:
        print(f'  [--]   {arquivo} não encontrado no FTP')
        return False
    except Exception as e:
        print(f'  [ERR]  {arquivo}: {e}')
        destino.unlink(missing_ok=True)
        return False

def _extrair_dbf_pop(zip_path, dbf_dest):
    if dbf_dest.exists():
        print(f'  [SKIP] {dbf_dest.name} (DBF já extraído)')
        return True
    dbf_dest.parent.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            dbfs = [n for n in z.namelist() if n.lower().endswith('.dbf')]
            if not dbfs:
                print(f'  [ERR]  {zip_path.name}: nenhum DBF no ZIP')
                return False
            with z.open(dbfs[0]) as src, open(dbf_dest, 'wb') as dst:
                dst.write(src.read())
        print(f'  [OK]   {dbf_dest.name}  ({dbf_dest.stat().st_size / 1024 / 1024:.1f} MB)')
        return True
    except Exception as e:
        print(f'  [ERR]  {zip_path.name}: {e}')
        dbf_dest.unlink(missing_ok=True)
        return False

def bronze_populacao(anos=ANOS):
    print(f'\n{"=" * 60}')
    print(f'Bronze — IBGE Projeções Populacionais')
    print(f'FTP     : {FTP_PATH_IBGE_POP}')
    print(f'Anos    : {anos[0]}–{anos[-1]}  ({len(anos)} exercícios)')
    print(f'{"=" * 60}\n')

    print('[ Etapa 1/4 — Download ZIP ]\n')
    # Só abre FTP se houver anos sem parquet nem zip local
    pendentes_ftp = [
        ano for ano in anos
        if not (IBGE_POP_BASE / 'parquet' / str(ano) / _nome_parquet_pop(ano)).exists()
        and not (IBGE_POP_BASE / 'zip' / str(ano) / _nome_zip_pop(ano)).exists()
    ]

    if not pendentes_ftp:
        print(f'  [SKIP] Todos os {len(anos)} Parquets já existem — FTP dispensado.\n')
    else:
        print(f'  {len(pendentes_ftp)} ano(s) pendente(s) de download.\n')
        ftp = conectar_ftp()
        for ano in pendentes_ftp:
            zip_dest = IBGE_POP_BASE / 'zip' / str(ano) / _nome_zip_pop(ano)
            _baixar_zip_pop(ftp, _nome_zip_pop(ano), zip_dest)
        try:
            ftp.quit()
        except OSError:
            ftp.close()
    print()

    print('[ Etapa 2/4 — Extração ZIP → DBF ]\n')
    for ano in anos:
        zip_path     = IBGE_POP_BASE / 'zip'     / str(ano) / _nome_zip_pop(ano)
        dbf_dest     = IBGE_POP_BASE / 'dbf'     / str(ano) / _nome_dbf_pop(ano)
        parquet_dest = IBGE_POP_BASE / 'parquet' / str(ano) / _nome_parquet_pop(ano)
        if parquet_dest.exists():
            print(f'  [SKIP] {_nome_dbf_pop(ano)} (Parquet já existe)')
            continue
        if zip_path.exists():
            _extrair_dbf_pop(zip_path, dbf_dest)
    print()

    print('[ Etapa 3/4 — Conversão DBF → Parquet ]\n')
    convertidos_parquet = 0
    for ano in anos:
        dbf_path     = IBGE_POP_BASE / 'dbf'     / str(ano) / _nome_dbf_pop(ano)
        parquet_path = IBGE_POP_BASE / 'parquet' / str(ano) / _nome_parquet_pop(ano)
        
        if not dbf_path.exists() and not parquet_path.exists():
            continue
        
        if dbf_para_parquet(dbf_path, parquet_path):
            convertidos_parquet += 1

    print('[ Etapa 4/4 — Conversão DBF → CSV ]\n')
    convertidos_csv = 0
    for ano in anos:
        dbf_path = IBGE_POP_BASE / 'dbf' / str(ano) / _nome_dbf_pop(ano)
        csv_path = IBGE_POP_BASE / 'csv' / str(ano) / _nome_csv_pop(ano)
        
        if not dbf_path.exists() and not csv_path.exists():
            continue
        
        if dbf_para_csv(dbf_path, csv_path):
            convertidos_csv += 1

    print(f'\nConcluído: {convertidos_parquet} arquivo(s) convertido(s).')
    print(f'Parquets em: {IBGE_POP_BASE / "parquet"}/\n')
    print(f'CSV em: {IBGE_POP_BASE / "csv"}/\n')

bronze_populacao(ANOS)


── Bronze 3/4: IBGE Projeções Populacionais ──────────────────────────


Bronze — IBGE Projeções Populacionais
FTP     : /dissemin/publicos/IBGE/POPSVS/
Anos    : 2010–2024  (15 exercícios)

[ Etapa 1/4 — Download ZIP ]

  [SKIP] Todos os 15 Parquets já existem — FTP dispensado.


[ Etapa 2/4 — Extração ZIP → DBF ]

  [SKIP] POP10.dbf (Parquet já existe)
  [SKIP] POP11.dbf (Parquet já existe)
  [SKIP] POP12.dbf (Parquet já existe)
  [SKIP] POP13.dbf (Parquet já existe)
  [SKIP] POP14.dbf (Parquet já existe)
  [SKIP] POP15.dbf (Parquet já existe)
  [SKIP] POP16.dbf (Parquet já existe)
  [SKIP] POP17.dbf (Parquet já existe)
  [SKIP] POP18.dbf (Parquet já existe)
  [SKIP] POP19.dbf (Parquet já existe)
  [SKIP] POP20.dbf (Parquet já existe)
  [SKIP] POP21.dbf (Parquet já existe)
  [SKIP] POP22.dbf (Parquet já existe)
  [SKIP] POP23.dbf (Parquet já existe)
  [SKIP] POP24.dbf (Parquet já existe)

[ Etapa 3/4 — Conversão DBF → Parquet ]

  [SKIP] POP10.parquet
  [SKIP] POP11.parquet
  [SKIP] 

### Bronze 4/4 — CID-10 (tabelas de referência)

Baixa as tabelas de referência CID-10 de duas fontes complementares:

- **v2008 (HTTP):** arquivo `CID10CSV.zip` do site DataSUS, contendo 6 CSVs com a hierarquia completa — capítulos, grupos, categorias, subcategorias e CID-O. O ZIP é excluído após extração.
- **FTP DataSUS:** arquivos `CID10.DBF` (categorias + subcategorias) e `CIDCAP10.DBF` (capítulos). Os DBFs são excluídos após conversão para CSV.

A variável `FONTE_CID10` (definida na célula de configuração) determina qual fonte será usada. O padrão é usar ambas, v2008 com fallback para FTP.


In [162]:
# ── Bronze 4/4: CID-10 ───────────────────────────────────────────────────────
# Baixa tabelas de referência CID-10 (fontes v2008 e/ou ftp).
# Arquivos intermediários (ZIP, DBF) são preservados no disco.
print(f'── Bronze 4/4: CID-10 ──────────────────────────\n')

def _limpar_cid(base, nome):
    print(f'  [AVISO] Falha no processamento de "{nome}": {base}')

def _baixar_zip_v2008(destino):
    if destino.exists():
        print(f'  [SKIP] {destino.name}')
        return True
    destino.parent.mkdir(parents=True, exist_ok=True)
    ctx = ssl._create_unverified_context()
    try:
        print(f'  Baixando {URL_CID10_V2008} ...')
        with urlopen(URL_CID10_V2008, timeout=30, context=ctx) as resp:
            destino.write_bytes(resp.read())
        print(f'  [OK]   {destino.name}  ({destino.stat().st_size / 1024:.1f} KB)')
        return True
    except Exception as e:
        print(f'  [ERR]  Download falhou: {e}')
        destino.unlink(missing_ok=True)
        return False

def _extrair_parquets_v2008(zip_path, parquet_dir):
    """Lê cada CSV do ZIP (latin-1, sep=;) e salva como Parquet."""
    parquet_dir.mkdir(parents=True, exist_ok=True)
    convertidos = 0
    with zipfile.ZipFile(zip_path, 'r') as z:
        membros = [n for n in z.namelist() if n.upper().endswith('.CSV')]
        for membro in membros:
            nome_base = Path(membro).name.upper().replace('.CSV', '')
            destino = parquet_dir / f'{nome_base}.parquet'
            if destino.exists():
                print(f'  [SKIP] {destino.name}')
                convertidos += 1
                continue
            try:
                raw = z.read(membro).decode('latin-1')
                df = pd.read_csv(io.StringIO(raw), sep=';', dtype=str)
                df = df.loc[:, ~df.columns.str.match(r'^Unnamed')]
                df.to_parquet(destino, index=False)
                df.to_csv(destino.with_suffix('.csv'), index=False, sep=';', encoding='utf-8-sig')
                print(f'  [OK]   {destino.name}  ({len(df):,} linhas)')
                convertidos += 1
            except Exception as e:
                print(f'  [ERR]  {destino.name}: {e}')
    return convertidos

def bronze_cid10_v2008():
    print(f'\n{"=" * 60}')
    print(f'Bronze — CID-10 (fonte: v2008)')
    print(f'URL     : {URL_CID10_V2008}')
    print(f'{"=" * 60}\n')
    zip_path    = CID_V2008_BASE / 'zip'     / 'CID10CSV.zip'
    parquet_dir = CID_V2008_BASE / 'parquet'

    print('[ Download ZIP ]\n')
    parquets_existentes = list(parquet_dir.glob('*.parquet')) if parquet_dir.exists() else []
    if len(parquets_existentes) >= 6:
        print(f'  [SKIP] {len(parquets_existentes)} Parquets já existem em {parquet_dir}')
    else:
        if not _baixar_zip_v2008(zip_path):
            _limpar_cid(CID_V2008_BASE, 'v2008')
            return
    print()
    print('[ Extração ZIP → Parquet + CSV ]\n')
    if not zip_path.exists() and not (parquet_dir.exists() and any(parquet_dir.glob('*.parquet'))):
        print(f'  [--] ZIP não encontrado e Parquets ausentes.')
        _limpar_cid(CID_V2008_BASE, 'v2008')
        return

    if zip_path.exists():
        try:
            n = _extrair_parquets_v2008(zip_path, parquet_dir)
        except Exception as e:
            print(f'  [ERR]  {e}')
            _limpar_cid(CID_V2008_BASE, 'v2008')
            return
        if n == 0:
            _limpar_cid(CID_V2008_BASE, 'v2008')
            return
    else:
        n = len(list(parquet_dir.glob('*.parquet')))
        print(f'  [SKIP] {n} Parquets já presentes.')

    print(f'\nConcluído: Parquets em: {parquet_dir}/\n')

def _baixar_ftp_cid(ftp, arquivo, destino):
    if destino.exists():
        print(f'  [SKIP] {arquivo}')
        return True
    destino.parent.mkdir(parents=True, exist_ok=True)
    try:
        ftp.cwd(FTP_PATH_CID)
        with open(destino, 'wb') as f:
            ftp.retrbinary(f'RETR {arquivo}', f.write)
        print(f'  [OK]   {arquivo}  ({destino.stat().st_size / 1024:.1f} KB)')
        return True
    except ftplib.error_perm:
        print(f'  [--]   {arquivo} não encontrado no FTP')
        return False
    except Exception as e:
        print(f'  [ERR]  {arquivo}: {e}')
        destino.unlink(missing_ok=True)
        return False

def bronze_cid10_ftp():
    print(f'\n{"=" * 60}')
    print(f'Bronze — CID-10 (fonte: ftp)')
    print(f'FTP     : ftp://{FTP_HOST}{FTP_PATH_CID}')
    print(f'{"=" * 60}\n')
    print('[ Download DBF ]\n')

    # Só conecta ao FTP se houver tabelas sem parquet nem dbf local
    pendentes_ftp = [
        arq for arq in CID_FTP_TABELAS
        if not (CID_FTP_BASE / 'parquet' / arq.replace('.DBF', '.parquet')).exists()
        and not (CID_FTP_BASE / 'dbf' / arq).exists()
    ]

    if not pendentes_ftp:
        print(f'  [SKIP] Todos os Parquets já existem — FTP dispensado.\n')
    else:
        try:
            ftp = conectar_ftp()
            falhas = sum(
                0 if _baixar_ftp_cid(ftp, arq, CID_FTP_BASE / 'dbf' / arq) else 1
                for arq in pendentes_ftp
            )
            ftp.quit()
        except Exception as e:
            print(f'  [ERR]  FTP: {e}')
            _limpar_cid(CID_FTP_BASE, 'ftp')
            return
        if falhas > 0:
            _limpar_cid(CID_FTP_BASE, 'ftp')
            return

    print()
    print('[ Conversão DBF → Parquet  (DBF excluído após conversão) ]\n')
    convertidos = 0
    for arquivo in CID_FTP_TABELAS:
        dbf_path     = CID_FTP_BASE / 'dbf'     / arquivo
        parquet_path = CID_FTP_BASE / 'parquet' / arquivo.replace('.DBF', '.parquet')
        if not dbf_path.exists() and not parquet_path.exists():
            continue
        if dbf_para_parquet(dbf_path, parquet_path):
            convertidos += 1

    if convertidos == 0:
        _limpar_cid(CID_FTP_BASE, 'ftp')
    else:
        print(f'\nConcluído: {convertidos} tabela(s). Parquets em: {CID_FTP_BASE / "parquet"}/\n')


def bronze_cid10(fonte=FONTE_CID10):
    fontes = [fonte] if fonte else ['v2008', 'ftp']
    for f in fontes:
        if f == 'v2008':
            bronze_cid10_v2008()
        elif f == 'ftp':
            bronze_cid10_ftp()

bronze_cid10(FONTE_CID10)


── Bronze 4/4: CID-10 ──────────────────────────


Bronze — CID-10 (fonte: v2008)
URL     : http://www2.datasus.gov.br/cid10/V2008/downloads/CID10CSV.zip

[ Download ZIP ]

  [SKIP] 6 Parquets já existem em dados/1-bronze/cid_10_datasus_v2008/parquet

[ Extração ZIP → Parquet + CSV ]

  [SKIP] CID-10-CATEGORIAS.parquet
  [SKIP] CID-10-CAPITULOS.parquet
  [SKIP] CID-10-GRUPOS.parquet
  [SKIP] CID-10-SUBCATEGORIAS.parquet
  [SKIP] CID-O-CATEGORIAS.parquet
  [SKIP] CID-O-GRUPOS.parquet

Concluído: Parquets em: dados/1-bronze/cid_10_datasus_v2008/parquet/


Bronze — CID-10 (fonte: ftp)
FTP     : ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/TABELAS/

[ Download DBF ]

  [SKIP] Todos os Parquets já existem — FTP dispensado.


[ Conversão DBF → Parquet  (DBF excluído após conversão) ]

  [SKIP] CID10.parquet
  [SKIP] CIDCAP10.parquet

Concluído: 2 tabela(s). Parquets em: dados/1-bronze/cid_10_datasus_ftp/parquet/



---

## ✅ Bronze — Resumo dos artefatos gerados

Após a execução bem-sucedida das quatro células acima, a estrutura de pastas conterá apenas Parquets:

```
dados/1-bronze/
├── SIM/
│   └── parquet/
│       ├── 2010/  DO{UF}2010.parquet  (até 27 arquivos)
│       ├── ...
│       └── 2024/  DO{UF}2024.parquet
├── ibge_dados_municipios/
│   └── parquet/
│       └── municipios.parquet
├── ibge_populacao/
│   └── parquet/
│       ├── 2010/  POP10.parquet
│       ├── ...
│       └── 2024/  POP24.parquet
├── cid_10_datasus_v2008/
│   └── parquet/
│       ├── CID-10-CAPITULOS.parquet
│       ├── CID-10-GRUPOS.parquet
│       ├── CID-10-CATEGORIAS.parquet
│       ├── CID-10-SUBCATEGORIAS.parquet
│       ├── CID-O-CATEGORIAS.parquet
│       └── CID-O-GRUPOS.parquet
└── cid_10_datasus_ftp/
    └── parquet/
        ├── CID10.parquet
        └── CIDCAP10.parquet
```

Os arquivos intermediários (`.dbc`, `.dbf`, `.zip`, `.json`) são excluídos automaticamente junto com suas pastas após a geração de cada Parquet.

> **Próximo passo:** executar o notebook da **camada Prata** para limpeza, normalização e integração dos dados.


---

## 🥈 PRATA — Limpeza, normalização e integração

Lê os Parquets da camada Bronze, normaliza e produz datasets prontos para análise.  
Nenhum dado é baixado nesta etapa — apenas transformação de conteúdo.

Execute as quatro células abaixo em ordem, ou use a célula **Prata — Execução principal** para rodar tudo de uma vez.

In [163]:
# ── Prata 1/4: CID-10 ────────────────────────────────────────────────────────
# Consolida as tabelas CID-10 do Bronze em um único dataset normalizado.
# Lê Parquets da camada Bronze (v2008 ou ftp, conforme FONTE_CID10).
# Schema: codigo_capitulo ; descricao_capitulo ; codigo_grupo ; descricao_grupo ;
#         codigo_categoria ; descricao_categoria ; codigo_cid10 ; descricao_cid10
print(f'── Prata 1/4: CID-10 ──────────────────────────\n')

P_CID10_V2008 = CID_V2008_BASE / 'parquet'
P_CID10_FTP   = CID_FTP_BASE   / 'parquet'
P_CID10_SAIDA = PRATA_DIR / 'CID10' / 'cid10.parquet'

CID10_COLS = [
    'codigo_capitulo', 'descricao_capitulo',
    'codigo_grupo',    'descricao_grupo',
    'codigo_categoria','descricao_categoria',
    'codigo_cid10',    'descricao_cid10',
]

_RE_DESC = re.compile(r'^[A-Z]\d+\.?\d*\s+')
_RE_CAP  = re.compile(r'^[IVXLCDM]+\.\s+')

def _ler_parquet_b(caminho):
    return pd.read_parquet(caminho).fillna('').astype(str)

def _strip_df(df):
    for col in df.select_dtypes(include=['object', 'str']).columns:
        df[col] = df[col].str.strip()
    return df

def _intervalo(cod, lookup):
    for *vals, ini, fim in lookup:
        if ini <= cod <= fim:
            return tuple(vals)
    return tuple('' for _ in lookup[0][:-2]) if lookup else ()

def _prata_cid10_v2008(pasta):
    print('\n[ Carregando v2008 ]\n')

    # Capítulos
    df_cap = _strip_df(_ler_parquet_b(pasta / 'CID-10-CAPITULOS.parquet')[['NUMCAP','CATINIC','CATFIM','DESCRICAO']])
    df_cap.columns = ['codigo_capitulo','catinic','catfim','descricao_capitulo']
    print(f'  [OK]  Capítulos:     {len(df_cap):>5}')

    # Grupo
    df_grp = _strip_df(_ler_parquet_b(pasta / 'CID-10-GRUPOS.parquet')[['CATINIC','CATFIM','DESCRICAO']])
    df_grp.columns = ['catinic','catfim','descricao_grupo']
    df_grp.insert(0, 'codigo_grupo', range(1, len(df_grp) + 1))
    print(f'  [OK]  Grupos:        {len(df_grp):>5}')

    # Categoria
    df_cat = _strip_df(_ler_parquet_b(pasta / 'CID-10-CATEGORIAS.parquet')[['CAT','DESCRICAO']])
    df_cat.columns = ['codigo_categoria','descricao_categoria']
    print(f'  [OK]  Categorias:    {len(df_cat):>5}')

    # SubCategoria
    df_sub = _strip_df(_ler_parquet_b(pasta / 'CID-10-SUBCATEGORIAS.parquet')[['SUBCAT','DESCRICAO']])
    df_sub.columns = ['codigo_cid10','descricao_cid10']
    print(f'  [OK]  Subcategorias: {len(df_sub):>5}')

    print('\n[ Consolidando ]\n')
    df_sub['codigo_categoria'] = df_sub['codigo_cid10'].str[:3]
    df_sub = df_sub.merge(df_cat, on='codigo_categoria', how='left')
    df_sub['descricao_categoria'] = df_sub['descricao_categoria'].fillna('')

    cap_lkp = list(df_cap[['codigo_capitulo','descricao_capitulo','catinic','catfim']].itertuples(index=False, name=None))
    grp_lkp = list(df_grp[['codigo_grupo','descricao_grupo','catinic','catfim']].itertuples(index=False, name=None))

    cap_s = df_sub['codigo_categoria'].apply(lambda c: pd.Series(_intervalo(c, cap_lkp), index=['codigo_capitulo','descricao_capitulo']))
    grp_s = df_sub['codigo_categoria'].apply(lambda c: pd.Series(_intervalo(c, grp_lkp), index=['codigo_grupo','descricao_grupo']))

    df_sub = pd.concat([cap_s, grp_s, df_sub], axis=1)
    print(f'  [OK]  {len(df_sub):,} linhas consolidadas.')
    return df_sub[CID10_COLS].copy()

def _prata_cid10_ftp(pasta):
    print('\n[ Carregando ftp ]\n')
    df_cap = _ler_parquet_b(pasta / 'CIDCAP10.parquet')
    df_cap.columns = ['desc_raw','causas']
    df_cap['codigo_capitulo']    = range(1, len(df_cap) + 1)
    df_cap['descricao_capitulo'] = df_cap['desc_raw'].str.strip().apply(lambda x: _RE_CAP.sub('', x).strip())
    df_cap['catinic'] = df_cap['causas'].str.split('-').str[0].str.strip()
    df_cap['catfim']  = df_cap['causas'].str.split('-').str[1].str.strip()
    print(f'  [OK]  Capítulos:     {len(df_cap):>5}')

    df = _ler_parquet_b(pasta / 'CID10.parquet')
    for col in ['CID10','CAT','SUBCAT','DESCR']:
        df[col] = df[col].str.strip()

    df_cat = df[df['CAT'] == 'S'][['CID10','DESCR']].copy()
    df_cat.columns = ['codigo_categoria','descr_raw']
    df_cat['descricao_categoria'] = df_cat['descr_raw'].apply(lambda x: _RE_DESC.sub('', x).strip())
    df_cat = df_cat[['codigo_categoria','descricao_categoria']]
    print(f'  [OK]  Categorias:    {len(df_cat):>5}')

    df_sub = df[df['SUBCAT'] == 'S'][['CID10','DESCR']].copy()
    df_sub.columns = ['codigo_cid10','descr_raw']
    df_sub['descricao_cid10'] = df_sub['descr_raw'].apply(lambda x: _RE_DESC.sub('', x).strip())
    print(f'  [OK]  Subcategorias: {len(df_sub):>5}')

    print('\n[ Consolidando ]\n')
    df_sub['codigo_categoria'] = df_sub['codigo_cid10'].str[:3]
    df_sub = df_sub.merge(df_cat, on='codigo_categoria', how='left')
    df_sub['descricao_categoria'] = df_sub['descricao_categoria'].fillna('')

    cap_lkp = list(df_cap[['codigo_capitulo','descricao_capitulo','catinic','catfim']].itertuples(index=False, name=None))
    cap_s = df_sub['codigo_categoria'].apply(lambda c: pd.Series(_intervalo(c, cap_lkp), index=['codigo_capitulo','descricao_capitulo']))

    df_sub = pd.concat([cap_s, df_sub], axis=1)
    df_sub['codigo_grupo']    = ''
    df_sub['descricao_grupo'] = ''
    print(f'  [OK]  {len(df_sub):,} linhas consolidadas.')
    return df_sub[CID10_COLS].copy()

def prata_cid10(fonte=FONTE_CID10):
    if fonte is not None:
        pasta = P_CID10_V2008 if fonte == 'v2008' else P_CID10_FTP
        if not pasta.exists():
            print(f'[ERRO] Pasta da fonte "{fonte}" não encontrada: {pasta}')
            print('[ERRO] Execute Bronze 4/4 antes desta célula.')
            return
        nome_fonte = fonte
    elif P_CID10_V2008.exists():
        pasta, nome_fonte = P_CID10_V2008, 'v2008'
    elif P_CID10_FTP.exists():
        print('[AVISO] v2008 não encontrada. Usando fallback: ftp.')
        pasta, nome_fonte = P_CID10_FTP, 'ftp'
    else:
        print('[ERRO] Nenhuma fonte CID-10 encontrada. Execute Bronze 4/4 primeiro.')
        return

    print(f'\n{"=" * 60}')
    print(f'Prata — CID-10  [{nome_fonte}]')
    print(f'Entrada : {pasta}')
    print(f'Saída   : {P_CID10_SAIDA}')
    print(f'{"=" * 60}')

    df = _prata_cid10_v2008(pasta) if nome_fonte == 'v2008' else _prata_cid10_ftp(pasta)

    P_CID10_SAIDA.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(P_CID10_SAIDA, index=False)
    df.to_csv(P_CID10_SAIDA.with_suffix('.csv'), index=False, sep=';', encoding='utf-8-sig')
    print(f'\n  [OK]  {len(df):,} linhas  →  {P_CID10_SAIDA}')
    print('\nConcluído.')

prata_cid10(FONTE_CID10)

── Prata 1/4: CID-10 ──────────────────────────


Prata — CID-10  [v2008]
Entrada : dados/1-bronze/cid_10_datasus_v2008/parquet
Saída   : dados/2-prata/CID10/cid10.parquet

[ Carregando v2008 ]

  [OK]  Capítulos:        22
  [OK]  Grupos:          275
  [OK]  Categorias:     2045
  [OK]  Subcategorias: 12451

[ Consolidando ]

  [OK]  12,451 linhas consolidadas.

  [OK]  12,451 linhas  →  dados/2-prata/CID10/cid10.parquet

Concluído.


In [164]:
# ── Prata 2/4: IBGE Municípios ───────────────────────────────────────────────
# Gera dataset enxuto de municípios com identificadores e hierarquia geográfica.
# Schema: codigo_municipio ; codigo_municipio_6c ; nome_municipio ;
#         codigo_estado ; sigla_estado ; nome_estado ;
#         codigo_regiao ; sigla_regiao ; nome_regiao
print(f'── Prata 2/4: IBGE Municípios ──────────────────────────\n')

P_MUN_BRONZE = IBGE_MUN_BASE / 'parquet' / 'municipios.parquet'
P_MUN_SAIDA  = PRATA_DIR / 'IBGE' / 'ibge_municipios.parquet'

_P_MUN_COLUNAS  = ['id','nome','uf_id','uf_sigla','uf_nome','regiao_id','regiao_sigla','regiao_nome']
_P_MUN_RENOMEAR = {
    'id'          : 'codigo_municipio',
    'nome'        : 'nome_municipio',
    'uf_id'       : 'codigo_estado',
    'uf_sigla'    : 'sigla_estado',
    'uf_nome'     : 'nome_estado',
    'regiao_id'   : 'codigo_regiao',
    'regiao_sigla': 'sigla_regiao',
    'regiao_nome' : 'nome_regiao',
}

def prata_municipios():
    print(f'\n{"=" * 60}')
    print(f'Prata — IBGE Municípios')
    print(f'Entrada : {P_MUN_BRONZE}')
    print(f'Saída   : {P_MUN_SAIDA}')
    print(f'{"=" * 60}\n')

    if not P_MUN_BRONZE.exists():
        print(f'[ERRO] Arquivo não encontrado: {P_MUN_BRONZE}')
        print('[ERRO] Execute Bronze 2/4 antes desta célula.')
        return

    # astype(object) converte Int64 nullable para object antes do fillna com string
    df = pd.read_parquet(P_MUN_BRONZE).astype(object).fillna('').astype(str)
    df = df[_P_MUN_COLUNAS].rename(columns=_P_MUN_RENOMEAR)
    for col in df.columns:
        df[col] = df[col].str.strip()
    df.insert(1, 'codigo_municipio_6c', df['codigo_municipio'].str[:6])

    print(f'  [OK]  {len(df):,} municípios carregados.')

    P_MUN_SAIDA.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(P_MUN_SAIDA, index=False)
    df.to_csv(P_MUN_SAIDA.with_suffix('.csv'), index=False, sep=';', encoding='utf-8-sig')
    print(f'  [OK]  {len(df):,} linhas  →  {P_MUN_SAIDA}')
    print('\nConcluído.')

prata_municipios()

── Prata 2/4: IBGE Municípios ──────────────────────────


Prata — IBGE Municípios
Entrada : dados/1-bronze/ibge_dados_municipios/parquet/municipios.parquet
Saída   : dados/2-prata/IBGE/ibge_municipios.parquet

  [OK]  5,571 municípios carregados.
  [OK]  5,571 linhas  →  dados/2-prata/IBGE/ibge_municipios.parquet

Concluído.


In [165]:
# ── Prata 3/4: IBGE Projeções Populacionais ─────────────────────────────────
# Agrega população por município, exercício e sexo (soma todas as faixas etárias).
# Schema: codigo_municipio ; exercicio ; sexo (M/F/I) ; populacao
print(f'── Prata 3/4: IBGE Projeções Populacionais ──────────────────────────\n')

P_POP_BRONZE = IBGE_POP_BASE / 'parquet'
P_POP_SAIDA  = PRATA_DIR / 'IBGE' / 'ibge_populacao.parquet'

_MAPA_SEXO_POP = {'1': 'M', '2': 'F', '0': 'I', '9': 'I'}

def prata_populacao(anos=None):
    print(f'\n{"=" * 60}')
    print(f'Prata — IBGE Projeções Populacionais')
    print(f'Entrada : {P_POP_BRONZE}/YYYY/*.parquet')
    print(f'Saída   : {P_POP_SAIDA}')
    if anos:
        print(f'Anos    : {anos}')
    print(f'{"=" * 60}')

    if not P_POP_BRONZE.exists():
        print(f'\n[ERRO] Diretório não encontrado: {P_POP_BRONZE}')
        print('[ERRO] Execute Bronze 3/4 antes desta célula.')
        return

    arquivos = []
    for pasta in sorted(p for p in P_POP_BRONZE.iterdir() if p.is_dir()):
        try:
            ano = int(pasta.name)
        except ValueError:
            continue
        if anos and ano not in anos:
            continue
        parquets = list(pasta.glob('*.parquet'))
        if parquets:
            arquivos.append((ano, parquets[0]))

    if not arquivos:
        print('\n[ERRO] Nenhum Parquet encontrado. Execute Bronze 3/4 primeiro.')
        return

    print(f'\n[ Carregando {len(arquivos)} exercício(s) ]\n')
    partes = []
    for ano, caminho in arquivos:
        df = pd.read_parquet(caminho)
        df['COD_MUN'] = df['COD_MUN'].astype(str)
        df['ANO']     = df['ANO'].astype(str)
        df['SEXO']    = df['SEXO'].astype(str)
        df['POP']     = pd.to_numeric(df['POP'], errors='coerce').fillna(0).astype(int)
        agg = df.groupby(['COD_MUN','ANO','SEXO'], sort=False)['POP'].sum().reset_index()
        agg['SEXO'] = agg['SEXO'].map(_MAPA_SEXO_POP).fillna('O')
        agg.rename(columns={'COD_MUN':'codigo_municipio','ANO':'exercicio',
                             'SEXO':'sexo','POP':'populacao'}, inplace=True)
        mun = agg['codigo_municipio'].nunique()
        pop = int(agg['populacao'].sum())
        print(f'  [OK]  {ano}  —  {mun:,} municípios  |  população total: {pop:,}')
        partes.append(agg)

    df_final = pd.concat(partes, ignore_index=True)
    df_final = df_final.sort_values(['exercicio','codigo_municipio','sexo']).reset_index(drop=True)
    ex  = df_final['exercicio'].nunique()
    mun = df_final['codigo_municipio'].nunique()
    print(f'\n  [OK]  Total: {len(df_final):,} linhas ({ex} exercícios × {mun:,} municípios × sexo)')

    P_POP_SAIDA.parent.mkdir(parents=True, exist_ok=True)
    df_final.to_parquet(P_POP_SAIDA, index=False)
    df_final.to_csv(P_POP_SAIDA.with_suffix('.csv'), index=False, sep=';', encoding='utf-8-sig')
    print(f'  [OK]  {len(df_final):,} linhas  →  {P_POP_SAIDA}')
    print('\nConcluído.')

prata_populacao()

── Prata 3/4: IBGE Projeções Populacionais ──────────────────────────


Prata — IBGE Projeções Populacionais
Entrada : dados/1-bronze/ibge_populacao/parquet/YYYY/*.parquet
Saída   : dados/2-prata/IBGE/ibge_populacao.parquet

[ Carregando 15 exercício(s) ]

  [OK]  2010  —  5,570 municípios  |  população total: 194,749,329
  [OK]  2011  —  5,570 municípios  |  população total: 196,158,610
  [OK]  2012  —  5,570 municípios  |  população total: 197,670,620
  [OK]  2013  —  5,570 municípios  |  população total: 199,226,702
  [OK]  2014  —  5,570 municípios  |  população total: 200,811,131
  [OK]  2015  —  5,570 municípios  |  população total: 202,403,642
  [OK]  2016  —  5,570 municípios  |  população total: 203,871,925
  [OK]  2017  —  5,570 municípios  |  população total: 205,211,557
  [OK]  2018  —  5,570 municípios  |  população total: 206,529,038
  [OK]  2019  —  5,570 municípios  |  população total: 207,900,099
  [OK]  2020  —  5,570 municípios  |  população total: 209,164,889
  [OK]

In [166]:
# ── Prata 4/4: DataSUS SIM — Declarações de Óbito ───────────────────────────
# Agrega óbitos por município, causa (CID-10), sexo e exercício.
# Filtros aplicados (em ordem):
#   1. TIPOBITO=2          — apenas óbitos não-fetais
#   2. CODMUNOCOR preenchido — município de ocorrência informado
#   3. CAUSABAS preenchido   — causa básica informada
#   4. CODMUNOCOR válido     — código IBGE existe na tabela de municípios Prata
#
# DCNT consideradas:
#   I00-I99            → doenças cardiovasculares
#   C00-C97            → neoplasias malignas
#   J30-J98 exceto J36 → doenças respiratórias crônicas
#   E10-E14            → diabetes mellitus
#
# Schema: sexo ; codigo_municipio ; cid10 ; exercicio ; dcnt ; arquivo_origem ; obitos
# Saída  : um arquivo por exercício  →  dados/2-prata/SIM/{ANO}/datasus_sim_{ANO}.parquet

P_SIM_BRONZE = SIM_BASE / 'parquet'

_MAPA_SEXO_SIM = {'1': 'M', '2': 'F', '0': 'I', '9': 'I'}

_SIM_RENOMEAR = {
    'SEXO'           : 'sexo',
    'CODMUNOCOR'     : 'codigo_municipio',
    'CAUSABAS'       : 'cid10',
    'EXERCICIO'      : 'exercicio',
    'DCNT'           : 'dcnt',
    'ARQUIVO_ORIGEM' : 'arquivo_origem',
    'QUANTIDADE'     : 'obitos',
}

def _e_dcnt(cid):
    if not isinstance(cid, str):
        return 'N'
    cid = cid.strip().upper()
    if len(cid) < 3:
        return 'N'
    letra = cid[0]
    try:
        num = int(cid[1:3])
    except ValueError:
        return 'N'
    if letra == 'I':                                    return 'S'
    if letra == 'C' and num <= 97:                      return 'S'
    if letra == 'J' and 30 <= num <= 98 and num != 36: return 'S'
    if letra == 'E' and 10 <= num <= 14:                return 'S'
    return 'N'

def prata_sim(anos=None):
    print(f'\n{"=" * 60}')
    print(f'Prata — DataSUS SIM (Declarações de Óbito)')
    print(f'Entrada : {P_SIM_BRONZE}/YYYY/*.parquet')
    print(f'Saída   : {PRATA_DIR}/SIM/{{ANO}}/datasus_sim_{{ANO}}.parquet')
    if anos:
        print(f'Anos    : {anos}')
    print(f'{"=" * 60}')

    if not P_SIM_BRONZE.exists():
        print(f'\n[ERRO] Diretório não encontrado: {P_SIM_BRONZE}')
        print('[ERRO] Execute Bronze 1/4 antes desta célula.')
        return

    p_mun = PRATA_DIR / 'IBGE' / 'ibge_municipios.parquet'
    if not p_mun.exists():
        print('[ERRO] ibge_municipios.parquet não encontrado. Execute Prata 2/4 antes desta célula.')
        return
    mun_validos = set(pd.read_parquet(p_mun)['codigo_municipio_6c'].astype(str))
    print(f'\n  Municípios IBGE válidos carregados: {len(mun_validos):,}')

    por_ano: dict[int, list] = {}
    for pasta in sorted(p for p in P_SIM_BRONZE.iterdir() if p.is_dir()):
        try:
            ano = int(pasta.name)
        except ValueError:
            continue
        if anos and ano not in anos:
            continue
        arquivos_ano = sorted(pasta.glob('*.parquet'))
        if arquivos_ano:
            por_ano[ano] = arquivos_ano

    if not por_ano:
        print('\n[ERRO] Nenhum Parquet encontrado. Execute Bronze 1/4 primeiro.')
        return

    P_SIM_PRATA = PRATA_DIR / 'SIM'
    P_SIM_PRATA.mkdir(parents=True, exist_ok=True)

    # Contadores globais para o resumo de exclusões
    g_total = g_rem_tipobito = g_rem_codmun = g_rem_cid = g_rem_mun_inv = 0

    for ano in sorted(por_ano):
        arquivos_ano = por_ano[ano]
        pasta_ano = PRATA_DIR / 'SIM' / str(ano)
        pasta_ano.mkdir(parents=True, exist_ok=True)
        saida_pq  = pasta_ano / f'datasus_sim_{ano}.parquet'
        saida_csv = pasta_ano / f'datasus_sim_{ano}.csv'

        print(f'\n[ Exercício {ano} — {len(arquivos_ano)} arquivo(s) ]')

        partes = []
        for caminho in arquivos_ano:
            try:
                df = pd.read_parquet(caminho)
            except Exception as e:
                print(f'  [ERR]  {caminho.name}: {e}')
                continue

            for col in ['TIPOBITO', 'CODMUNOCOR', 'CAUSABAS', 'SEXO']:
                df[col] = df[col].astype(str).str.strip()

            n_bruto = len(df)
            g_total += n_bruto

            df = df[df['TIPOBITO'] == '2']
            g_rem_tipobito += n_bruto - len(df)

            n_apos_tipo = len(df)
            df = df[df['CODMUNOCOR'] != '']
            g_rem_codmun += n_apos_tipo - len(df)

            n_apos_mun = len(df)
            mask_cid = (df['CAUSABAS'] == '') | (df['CAUSABAS'] == 'nan')
            g_rem_cid += int(mask_cid.sum())
            df = df[~mask_cid]

            n_apos_cid = len(df)
            mask_inv = ~df['CODMUNOCOR'].isin(mun_validos)
            g_rem_mun_inv += int(mask_inv.sum())
            df = df[~mask_inv]

            if df.empty:
                print(f'  [--]   {caminho.name}  (sem registros após filtros)')
                continue

            df['SEXO']           = df['SEXO'].map(_MAPA_SEXO_SIM).fillna('I')
            df['EXERCICIO']      = str(ano)
            df['DCNT']           = df['CAUSABAS'].apply(_e_dcnt)
            df['ARQUIVO_ORIGEM'] = caminho.stem

            agg = (df.groupby(['SEXO', 'CODMUNOCOR', 'CAUSABAS', 'EXERCICIO', 'DCNT', 'ARQUIVO_ORIGEM'], sort=False)
                     .size()
                     .reset_index(name='QUANTIDADE'))
            print(f'  [OK]   {caminho.name}  —  {int(agg["QUANTIDADE"].sum()):,} óbitos  ({len(agg):,} grupos)')
            partes.append(agg)

        if not partes:
            print(f'  [AVISO] Nenhum dado processado para {ano}.')
            continue

        df_ano = (pd.concat(partes, ignore_index=True)
                    .groupby(['SEXO', 'CODMUNOCOR', 'CAUSABAS', 'EXERCICIO', 'DCNT', 'ARQUIVO_ORIGEM'], sort=False)['QUANTIDADE']
                    .sum()
                    .reset_index()
                    .sort_values(['CODMUNOCOR', 'CAUSABAS', 'SEXO'])
                    .reset_index(drop=True)
                    .rename(columns=_SIM_RENOMEAR))

        total      = int(df_ano['obitos'].sum())
        total_dcnt = int(df_ano[df_ano['dcnt'] == 'S']['obitos'].sum())
        print(f'  [OK]  {total:,} óbitos  |  DCNT: {total_dcnt:,} ({total_dcnt / total * 100:.1f}%)')

        df_ano.to_parquet(saida_pq, index=False)
        df_ano.to_csv(saida_csv, index=False, sep=';', encoding='utf-8-sig')
        print(f'  [OK]  {len(df_ano):,} linhas  →  {saida_pq.name}')

    # ── Resumo de exclusões ──────────────────────────────────────────────────
    print(f'\n{"=" * 60}')
    print(f'Resumo de exclusões (total de registros brutos: {g_total:,})')
    print(f'{"=" * 60}')
    filtros = [
        ('TIPOBITO ≠ 2  (óbitos fetais)',          g_rem_tipobito),
        ('CODMUNOCOR vazio',                        g_rem_codmun),
        ('CAUSABAS vazio  (CID-10 ausente)',        g_rem_cid),
        ('CODMUNOCOR inválido  (não consta no IBGE)', g_rem_mun_inv),
    ]
    g_rem_total = sum(v for _, v in filtros)
    for label, n in filtros:
        pct = n / g_total * 100 if g_total else 0
        print(f'  Removidos — {label:<45} {n:>8,}  ({pct:.3f}%)')
    pct_final = (g_total - g_rem_total) / g_total * 100 if g_total else 0
    print(f'  {"─" * 62}')
    print(f'  Aproveitados após todos os filtros: {g_total - g_rem_total:,}  ({pct_final:.2f}%)')
    print('\nConcluído.')

prata_sim()


Prata — DataSUS SIM (Declarações de Óbito)
Entrada : dados/1-bronze/SIM/parquet/YYYY/*.parquet
Saída   : dados/2-prata/SIM/{ANO}/datasus_sim_{ANO}.parquet

  Municípios IBGE válidos carregados: 5,571

[ Exercício 2010 — 27 arquivo(s) ]
  [OK]   DOAC2010.parquet  —  3,009 óbitos  (1,124 grupos)
  [OK]   DOAL2010.parquet  —  17,737 óbitos  (5,324 grupos)
  [OK]   DOAM2010.parquet  —  13,300 óbitos  (3,832 grupos)
  [OK]   DOAP2010.parquet  —  2,172 óbitos  (783 grupos)
  [OK]   DOBA2010.parquet  —  76,309 óbitos  (27,143 grupos)
  [OK]   DOCE2010.parquet  —  43,847 óbitos  (15,129 grupos)
  [OK]   DODF2010.parquet  —  10,851 óbitos  (1,688 grupos)
  [OK]   DOES2010.parquet  —  21,205 óbitos  (8,136 grupos)
  [OK]   DOGO2010.parquet  —  32,655 óbitos  (11,254 grupos)
  [OK]   DOMA2010.parquet  —  26,091 óbitos  (10,775 grupos)
  [OK]   DOMG2010.parquet  —  120,802 óbitos  (46,512 grupos)
  [OK]   DOMS2010.parquet  —  14,471 óbitos  (5,935 grupos)
  [OK]   DOMT2010.parquet  —  14,986 óbit

---

## ✅ Prata — Resumo dos artefatos gerados

Após a execução bem-sucedida das quatro células acima, a estrutura de pastas conterá:

```
dados/2-prata/
├── CID10/
│   ├── cid10.parquet
│   └── cid10.csv
├── IBGE/
│   ├── ibge_municipios.parquet
│   ├── ibge_municipios.csv
│   ├── ibge_populacao.parquet
│   └── ibge_populacao.csv
└── SIM/
    ├── 2010/
    │   ├── datasus_sim_2010.parquet
    │   └── datasus_sim_2010.csv
    ├── ···
    └── 2020/
        ├── datasus_sim_2020.parquet
        └── datasus_sim_2020.csv
```

Filtros aplicados na camada SIM: `TIPOBITO=2` (óbito não-fetal), `CODMUNOCOR` preenchido, `CAUSABAS` preenchido.

> **Próximo passo:** executar a **camada Ouro** para enriquecimento e integração dos datasets.

---

## 🥇 OURO — Dataset analítico final

Combina os datasets Prata (SIM, Municípios, CID-10 e Populacao) em datasets enriquecidos, prontos para análise exploratória e modelagem.

Execute as duas células abaixo em ordem:

| Célula | Descrição | Saída |
|--------|-----------|-------|
| Ouro 1/2 | SIM × Municípios × CID-10 | `dados/3-ouro/SIM/{ANO}/sim_ouro_{ANO}.parquet` |
| Ouro 2/2 | Populacao × Municípios | `dados/3-ouro/IBGE/populacao_municipio.parquet` |

In [167]:
# ── Ouro 1/2: SIM × Municípios × CID-10 ─────────────────────────────────────
# Joins: sim.codigo_municipio = mun.codigo_municipio_6c (IBGE 6 dígitos)
#         sim.cid10 = cid.codigo_cid10
# Todos os municípios da Prata são válidos (filtro aplicado em Prata 4/4).
# Schema: sexo ; codigo_municipio ; codigo_municipio_6c ; nome_municipio ;
#         sigla_estado ; nome_estado ; sigla_regiao ; nome_regiao ;
#         codigo_cid10 ; descricao_cid10 ; exercicio ; dcnt ; arquivo_origem ; obitos
# Saídas: SIM/{ANO}/sim_ouro_{ANO}.parquet + sim_ouro_{ANO}.csv (sep=;, utf-8-sig)
print(f'── Ouro 1/2: SIM × Municípios × CID-10 ──────────────────────────\n')

P_CID_PRATA = PRATA_DIR / 'CID10' / 'cid10.parquet'
P_MUN_PRATA = PRATA_DIR / 'IBGE' / 'ibge_municipios.parquet'

_OURO_COLUNAS = [
    'sexo', 'codigo_municipio', 'codigo_municipio_6c', 'nome_municipio',
    'sigla_estado', 'nome_estado', 'sigla_regiao', 'nome_regiao',
    'codigo_cid10', 'descricao_cid10', 'exercicio', 'dcnt', 'arquivo_origem', 'obitos',
]

def ouro_sim():
    print(f'\n{"=" * 60}')
    print(f'Ouro — SIM × Municípios × CID-10')
    print(f'Saída   : {OURO_DIR}/SIM/{{ANO}}/')
    print(f'{"=" * 60}\n')

    for path, label in [
        (P_CID_PRATA, 'cid10.parquet'),
        (P_MUN_PRATA, 'ibge_municipios.parquet'),
    ]:
        if not path.exists():
            print(f'[ERRO] {label} não encontrado. Execute a camada Prata primeiro.')
            return

    arquivos_prata = sorted((PRATA_DIR / 'SIM').glob('*/datasus_sim_*.parquet'))
    if not arquivos_prata:
        print('[ERRO] Nenhum arquivo encontrado em dados/2-prata/SIM/. Execute Prata 4/4 primeiro.')
        return

    print('[ Carregando tabelas de referência ]\n')
    cid = pd.read_parquet(P_CID_PRATA)[['codigo_cid10', 'descricao_cid10']]
    mun = pd.read_parquet(P_MUN_PRATA)[
        ['codigo_municipio_6c', 'nome_municipio', 'sigla_estado', 'nome_estado', 'sigla_regiao', 'nome_regiao']
    ]
    print(f'  [OK]  CID-10     : {len(cid):,} linhas')
    print(f'  [OK]  Municípios : {len(mun):,} linhas')

    for prata_pq in arquivos_prata:
        ano = prata_pq.stem.replace('datasus_sim_', '')
        pasta_ano = OURO_DIR / 'SIM' / str(ano)
        pasta_ano.mkdir(parents=True, exist_ok=True)
        saida_pq  = pasta_ano / f'sim_ouro_{ano}.parquet'
        saida_csv = pasta_ano / f'sim_ouro_{ano}.csv'

        print(f'\n[ Exercício {ano} ]')
        sim = pd.read_parquet(prata_pq)
        print(f'  [OK]  SIM        : {len(sim):,} linhas')

        # SIM × Municípios
        df = sim.merge(mun, left_on='codigo_municipio', right_on='codigo_municipio_6c', how='left')
        sem_mun = int(df['codigo_municipio_6c'].isna().sum())
        if sem_mun:
            print(f'  [ERRO] {sem_mun:,} linhas sem correspondência em Municípios — verifique Prata 4/4')

        # × CID-10
        df = df.merge(cid, left_on='cid10', right_on='codigo_cid10', how='left')

        df = (df.drop(columns=['cid10'], errors='ignore')[_OURO_COLUNAS]
                .sort_values(['codigo_municipio', 'codigo_cid10', 'sexo'])
                .reset_index(drop=True))

        df.to_parquet(saida_pq, index=False)
        df.to_csv(saida_csv, index=False, sep=';', encoding='utf-8-sig')

        total      = int(df['obitos'].sum())
        total_dcnt = int(df[df['dcnt'] == 'S']['obitos'].sum())
        print(f'  [OK]  {len(df):,} linhas  ×  {len(df.columns)} colunas')
        print(f'  [OK]  Óbitos: {total:,}  |  DCNT: {total_dcnt:,} ({total_dcnt / total * 100:.1f}%)')
        print(f'  [OK]  {saida_pq.name}  ({saida_pq.stat().st_size / 1024 / 1024:.1f} MB)')

    print('\nConcluído.')

ouro_sim()

── Ouro 1/2: SIM × Municípios × CID-10 ──────────────────────────


Ouro — SIM × Municípios × CID-10
Saída   : dados/3-ouro/SIM/{ANO}/

[ Carregando tabelas de referência ]

  [OK]  CID-10     : 12,451 linhas
  [OK]  Municípios : 5,571 linhas

[ Exercício 2010 ]
  [OK]  SIM        : 360,040 linhas
  [OK]  360,040 linhas  ×  14 colunas
  [OK]  Óbitos: 1,136,906  |  DCNT: 619,804 (54.5%)
  [OK]  sim_ouro_2010.parquet  (1.8 MB)

[ Exercício 2011 ]
  [OK]  SIM        : 368,058 linhas
  [OK]  368,058 linhas  ×  14 colunas
  [OK]  Óbitos: 1,170,482  |  DCNT: 640,137 (54.7%)
  [OK]  sim_ouro_2011.parquet  (1.8 MB)

[ Exercício 2012 ]
  [OK]  SIM        : 373,492 linhas
  [OK]  373,492 linhas  ×  14 colunas
  [OK]  Óbitos: 1,181,122  |  DCNT: 642,530 (54.4%)
  [OK]  sim_ouro_2012.parquet  (1.8 MB)

[ Exercício 2013 ]
  [OK]  SIM        : 383,832 linhas
  [OK]  383,832 linhas  ×  14 colunas
  [OK]  Óbitos: 1,210,430  |  DCNT: 658,723 (54.4%)
  [OK]  sim_ouro_2013.parquet  (1.9 MB)

[ Exercício 

In [168]:
# ── Ouro 2/2: Populacao × Municípios ────────────────────────────────────────
# Enriquece a população Prata com a hierarquia geográfica completa dos municípios.
# Chave de join: pop.codigo_municipio (6c) = mun.codigo_municipio_6c
#
# Schema: exercicio ; codigo_municipio_6c ; codigo_municipio ; nome_municipio ;
#         sigla_estado ; nome_estado ; sigla_regiao ; nome_regiao ;
#         sexo ; populacao
# Saída : dados/3-ouro/IBGE/populacao_municipio.parquet  +  .csv
print(f'── Ouro 2/2: Populacao × Municípios ────────────────────────────────\n')

P_POP_PRATA  = PRATA_DIR / 'IBGE' / 'ibge_populacao.parquet'
P_MUN_PRATA  = PRATA_DIR / 'IBGE' / 'ibge_municipios.parquet'
P_POP_OURO   = OURO_DIR  / 'IBGE' / 'populacao_municipio.parquet'

_POP_COLUNAS = [
    'exercicio', 'codigo_municipio_6c', 'codigo_municipio',
    'nome_municipio', 'sigla_estado', 'nome_estado', 'sigla_regiao', 'nome_regiao',
    'sexo', 'populacao',
]

def ouro_populacao():
    print(f'\n{"=" * 60}')
    print(f'Ouro — Populacao × Municípios')
    print(f'Saída   : {P_POP_OURO}')
    print(f'{"=" * 60}\n')

    for p, label in [(P_POP_PRATA, 'ibge_populacao.parquet'),
                     (P_MUN_PRATA, 'ibge_municipios.parquet')]:
        if not p.exists():
            print(f'[ERRO] {label} não encontrado. Execute a camada Prata primeiro.')
            return

    pop = pd.read_parquet(P_POP_PRATA)
    # codigo_municipio na população é 6 dígitos (POPSVS usa código de 6c)
    pop['codigo_municipio'] = pop['codigo_municipio'].astype(str).str[:6]

    mun = pd.read_parquet(P_MUN_PRATA)[
        ['codigo_municipio_6c', 'codigo_municipio', 'nome_municipio',
         'sigla_estado', 'nome_estado', 'sigla_regiao', 'nome_regiao']
    ].rename(columns={'codigo_municipio': 'codigo_municipio_7c'})

    print(f'  [OK]  Populacao  : {len(pop):,} linhas  ({pop["exercicio"].nunique()} exercícios)')
    print(f'  [OK]  Municípios : {len(mun):,} linhas')

    df = pop.merge(mun, left_on='codigo_municipio', right_on='codigo_municipio_6c', how='left')

    sem_mun = int(df['codigo_municipio_6c'].isna().sum())
    if sem_mun:
        print(f'  [ERRO] {sem_mun:,} linhas sem correspondência em Municípios — verifique Prata 2/4')

    df = (df.drop(columns=['codigo_municipio'])          # remove 6c da pop (já em codigo_municipio_6c)
            .rename(columns={'codigo_municipio_7c': 'codigo_municipio'})  # promove 7c
            [_POP_COLUNAS]
            .sort_values(['exercicio', 'sigla_estado', 'codigo_municipio_6c', 'sexo'])
            .reset_index(drop=True))

    P_POP_OURO.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(P_POP_OURO, index=False)
    df.to_csv(P_POP_OURO.with_suffix('.csv'), index=False, sep=';', encoding='utf-8-sig')

    ex  = df['exercicio'].nunique()
    mun_n = df['codigo_municipio_6c'].nunique()
    print(f'\n  [OK]  {len(df):,} linhas  ({ex} exercícios × {mun_n:,} municípios × sexo)')
    print(f'  [OK]  {P_POP_OURO.name}  ({P_POP_OURO.stat().st_size / 1024 / 1024:.1f} MB)')
    print('\nConcluído.')

ouro_populacao()

── Ouro 2/2: Populacao × Municípios ────────────────────────────────


Ouro — Populacao × Municípios
Saída   : dados/3-ouro/IBGE/populacao_municipio.parquet

  [OK]  Populacao  : 167,100 linhas  (15 exercícios)
  [OK]  Municípios : 5,571 linhas

  [OK]  167,100 linhas  (15 exercícios × 5,570 municípios × sexo)
  [OK]  populacao_municipio.parquet  (1.4 MB)

Concluído.


---

## ✅ Ouro — Resumo dos artefatos gerados

Após a execução bem-sucedida das duas células acima, a estrutura de pastas conterá:

```
dados/3-ouro/
├── SIM/
│   ├── 2010/
│   │   ├── sim_ouro_2010.parquet
│   │   └── sim_ouro_2010.csv
│   ├── ···
│   ├── 2020/
│   │   ├── sim_ouro_2020.parquet
│   │   └── sim_ouro_2020.csv
│   └── taxa_dcnt_municipio.parquet / .csv   ← atributo derivado
└── IBGE/
    ├── populacao_municipio.parquet
    └── populacao_municipio.csv
```

**Ouro SIM** — colunas: `sexo`, `codigo_municipio`, `codigo_municipio_6c`, `nome_municipio`, `sigla_estado`, `nome_estado`, `sigla_regiao`, `nome_regiao`, `codigo_cid10`, `descricao_cid10`, `exercicio`, `dcnt`, `arquivo_origem`, `obitos`

**Ouro Populacao** — colunas: `exercicio`, `codigo_municipio_6c`, `codigo_municipio`, `nome_municipio`, `sigla_estado`, `nome_estado`, `sigla_regiao`, `nome_regiao`, `sexo`, `populacao`

---

## 📋 Semana 3 — Validação, Dicionário de Dados e Atributos Derivados

Entrega do marco da **Semana 3 (11/06–22/06)** conforme cronograma:

- **Validação da integração** — cobertura municipal, conferência de totais e registro de avisos.
- **Dicionário de dados** — nome, tipo e descrição de todas as colunas do dataset Ouro.
- **Atributo derivado** — taxa de mortalidade por DCNT por 100 mil habitantes, por município e exercício.

In [169]:
# ── Semana 3 / Validação da Integração ───────────────────────────────────────
# Carrega todos os arquivos Ouro e verifica cobertura municipal, conferência
# de totais entre Prata e Ouro, e distribuição DCNT × não-DCNT.
print('── Validação da Integração ──────────────────────────────────────\n')

import pandas as pd
from pathlib import Path

_ouro_files  = sorted((OURO_DIR / 'SIM').glob('*/sim_ouro_*.parquet'))
_prata_files = sorted((PRATA_DIR / 'SIM').glob('*/datasus_sim_*.parquet'))

if not _ouro_files:
    print('[ERRO] Nenhum arquivo Ouro encontrado. Execute Ouro 1/1 primeiro.')
else:
    # Carrega tudo
    _df_ouro  = pd.concat([pd.read_parquet(f) for f in _ouro_files],  ignore_index=True)
    _df_prata = pd.concat([pd.read_parquet(f) for f in _prata_files], ignore_index=True)

    _anos        = sorted(_df_ouro['exercicio'].unique())
    _n_mun_ouro  = _df_ouro['codigo_municipio'].nunique()
    _n_mun_ibge  = len(pd.read_parquet(PRATA_DIR / 'IBGE' / 'ibge_municipios.parquet'))
    _obitos_prata = int(_df_prata['obitos'].sum())
    _obitos_ouro  = int(_df_ouro['obitos'].sum())
    _nao_id       = int(_df_ouro[_df_ouro['nome_municipio'] == 'NÃO IDENTIFICADO']['obitos'].sum())
    _dcnt         = int(_df_ouro[_df_ouro['dcnt'] == 'S']['obitos'].sum())
    _total        = _obitos_ouro

    print(f'Exercícios cobertos : {_anos[0]}–{_anos[-1]}  ({len(_anos)} anos)')
    print(f'Municípios únicos   : {_n_mun_ouro:,}  /  {_n_mun_ibge:,} cadastrados IBGE'
          f'  ({_n_mun_ouro / _n_mun_ibge * 100:.1f}% de cobertura)')
    print()
    print(f'Óbitos na Prata     : {_obitos_prata:,}')
    print(f'Óbitos no Ouro      : {_obitos_ouro:,}')
    _diff = _obitos_prata - _obitos_ouro
    print(f'Diferença           : {_diff:,}'
          + (f'  ← registros sem CID-10 removidos na Prata' if _diff > 0 else '  ✓ consistente'))
    print()
    print(f'Óbitos DCNT         : {_dcnt:,}  ({_dcnt / _total * 100:.1f}%)')
    print(f'Óbitos não-DCNT     : {_total - _dcnt:,}  ({(_total - _dcnt) / _total * 100:.1f}%)')
    print()
    print(f'Mun. NÃO IDENTIFICADO: {_nao_id:,} óbitos  ({_nao_id / _total * 100:.2f}%)')
    print()
    print('Cobertura por exercício:')
    for _ano in _anos:
        _d = _df_ouro[_df_ouro['exercicio'] == _ano]
        print(f'  {_ano}  —  {int(_d["obitos"].sum()):>10,} óbitos  |  '
              f'{_d["codigo_municipio"].nunique():,} municípios')
    print('\n✓ Validação concluída.')

── Validação da Integração ──────────────────────────────────────

Exercícios cobertos : 2010–2024  (15 anos)
Municípios únicos   : 5,570  /  5,571 cadastrados IBGE  (100.0% de cobertura)

Óbitos na Prata     : 20,331,573
Óbitos no Ouro      : 20,331,573
Diferença           : 0  ✓ consistente

Óbitos DCNT         : 10,665,631  (52.5%)
Óbitos não-DCNT     : 9,665,942  (47.5%)

Mun. NÃO IDENTIFICADO: 0 óbitos  (0.00%)

Cobertura por exercício:
  2010  —   1,136,906 óbitos  |  5,531 municípios
  2011  —   1,170,482 óbitos  |  5,533 municípios
  2012  —   1,181,122 óbitos  |  5,541 municípios
  2013  —   1,210,430 óbitos  |  5,539 municípios
  2014  —   1,227,003 óbitos  |  5,539 municípios
  2015  —   1,264,141 óbitos  |  5,540 municípios
  2016  —   1,309,743 óbitos  |  5,557 municípios
  2017  —   1,312,637 óbitos  |  5,550 municípios
  2018  —   1,316,700 óbitos  |  5,549 municípios
  2019  —   1,349,784 óbitos  |  5,555 municípios
  2020  —   1,556,803 óbitos  |  5,554 municípios
  20

### 📖 Dicionário de Dados — Dataset Ouro (`sim_ouro_{ANO}`)

Arquivo analítico final gerado em `dados/3-ouro/SIM/{ANO}/sim_ouro_{ANO}.parquet`.  
Granularidade: **uma linha por combinação de** município × CID-10 × sexo × arquivo de origem.

| # | Coluna | Tipo | Origem | Descrição |
|---|--------|------|--------|-----------|
| 1 | `sexo` | string | SIM | Sexo do falecido: `M` = masculino · `F` = feminino · `I` = ignorado |
| 2 | `codigo_municipio` | string | SIM | Código IBGE de 6 dígitos do município de ocorrência do óbito (campo CODMUNOCOR) |
| 3 | `codigo_municipio_6c` | string | Municípios | Código IBGE de 6 dígitos validado na tabela de municípios; igual a `codigo_municipio` quando o município não é encontrado |
| 4 | `nome_municipio` | string | Municípios | Nome oficial do município; `NÃO IDENTIFICADO` quando o código IBGE não consta na tabela de referência |
| 5 | `sigla_estado` | string | Municípios | Sigla da Unidade Federativa (ex.: `PB`, `SP`); inferida pelos 2 primeiros dígitos do código IBGE quando o município não é encontrado |
| 6 | `nome_estado` | string | Municípios | Nome completo da UF (ex.: `Paraíba`) |
| 7 | `sigla_regiao` | string | Municípios | Sigla da grande região (ex.: `NE`, `SE`) |
| 8 | `nome_regiao` | string | Municípios | Nome da grande região (ex.: `Nordeste`) |
| 9 | `codigo_cid10` | string | CID-10 | Código da causa básica do óbito conforme CID-10 (ex.: `I219`) |
| 10 | `descricao_cid10` | string | CID-10 | Descrição textual da causa básica (ex.: `Infarto agudo do miocárdio não especificado`) |
| 11 | `exercicio` | string | SIM | Ano de ocorrência do óbito |
| 12 | `dcnt` | string | Calculado | Classificação DCNT: `S` = causa pertence às DCNT prioritárias · `N` = demais causas |
| 13 | `arquivo_origem` | string | SIM | Nome do arquivo bronze que originou o registro (ex.: `DOSP2010` = São Paulo 2010) |
| 14 | `obitos` | int | SIM | Contagem de óbitos agregada por grupo |

#### Critério de classificação DCNT (`dcnt = 'S'`)

| Faixa CID-10 | Grupo |
|---|---|
| I00–I99 | Doenças cardiovasculares |
| C00–C97 | Neoplasias malignas |
| J30–J98 (exceto J36) | Doenças respiratórias crônicas |
| E10–E14 | Diabetes mellitus |

#### Filtros aplicados na camada Prata (SIM)

| Filtro | Campo | Valor removido |
|---|---|---|
| Apenas óbitos não-fetais | `TIPOBITO` | ≠ `2` |
| Município de ocorrência preenchido | `CODMUNOCOR` | vazio ou `nan` |
| Causa básica preenchida | `CAUSABAS` | vazio ou `nan` |

In [170]:
# ── Semana 3 / Atributo Derivado: Taxa de Mortalidade por DCNT por 100 mil hab.
# Fórmula : taxa_100k = (obitos / populacao) × 100.000
# Agregação: município × exercício  (soma todas as causas DCNT, ambos os sexos)
# Saída    : dados/3-ouro/SIM/taxa_dcnt_municipio.parquet  +  .csv
print('── Taxa de Mortalidade DCNT por 100 mil hab. ────────────────────\n')

P_POP    = OURO_DIR  / 'IBGE' / 'populacao_municipio.parquet'
P_TAXA   = OURO_DIR  / 'SIM'  / 'taxa_dcnt_municipio.parquet'

if not P_POP.exists():
    print('[ERRO] populacao_municipio.parquet não encontrado. Execute Ouro 2/2 primeiro.')
else:
    _ouro_files = sorted((OURO_DIR / 'SIM').glob('*/sim_ouro_*.parquet'))
    if not _ouro_files:
        print('[ERRO] Nenhum arquivo Ouro encontrado. Execute Ouro 1/1 primeiro.')
    else:
        # Agrega óbitos DCNT por município + exercício (soma sexos)
        _df = pd.concat([pd.read_parquet(f) for f in _ouro_files], ignore_index=True)
        _obitos = (_df[_df['dcnt'] == 'S']
                     .groupby(['codigo_municipio', 'exercicio'], sort=False)['obitos']
                     .sum()
                     .reset_index()
                     .rename(columns={'obitos': 'obitos_dcnt'}))

        # Agrega população por município + exercício (soma sexos)
        # populacao_municipio.parquet já usa codigo_municipio_6c como chave
        _pop = (pd.read_parquet(P_POP)
                  .groupby(['codigo_municipio_6c', 'exercicio'], sort=False)['populacao']
                  .sum()
                  .reset_index()
                  .rename(columns={'codigo_municipio_6c': 'codigo_municipio'}))

        _taxa = _obitos.merge(_pop, on=['codigo_municipio', 'exercicio'], how='left')
        _taxa['taxa_dcnt_100k'] = (_taxa['obitos_dcnt'] / _taxa['populacao'] * 100_000).round(2)

        # Enriquece com nome e UF (do primeiro arquivo Ouro disponível como lookup)
        _lookup = (_df[['codigo_municipio', 'nome_municipio', 'sigla_estado',
                         'nome_estado', 'sigla_regiao', 'nome_regiao']]
                     .drop_duplicates('codigo_municipio'))
        _taxa = (_taxa.merge(_lookup, on='codigo_municipio', how='left')
                      [['exercicio', 'codigo_municipio', 'nome_municipio',
                        'sigla_estado', 'nome_estado', 'sigla_regiao', 'nome_regiao',
                        'obitos_dcnt', 'populacao', 'taxa_dcnt_100k']]
                      .sort_values(['exercicio', 'taxa_dcnt_100k'], ascending=[True, False])
                      .reset_index(drop=True))

        sem_pop = int(_taxa['populacao'].isna().sum())
        if sem_pop:
            print(f'  [AVISO] {sem_pop:,} municípios sem dado populacional — taxa ficará NaN')

        (OURO_DIR / 'SIM').mkdir(parents=True, exist_ok=True)
        _taxa.to_parquet(P_TAXA, index=False)
        _taxa.to_csv(P_TAXA.with_suffix('.csv'), index=False, sep=';', encoding='utf-8-sig')

        print(f'  [OK]  {len(_taxa):,} linhas  →  {P_TAXA.name}')
        print(f'\nTop 10 municípios por taxa DCNT/100k (exercício mais recente):')
        _ano_max = _taxa['exercicio'].max()
        print(_taxa[_taxa['exercicio'] == _ano_max]
              [['nome_municipio','sigla_estado','obitos_dcnt','populacao','taxa_dcnt_100k']]
              .head(10).to_string(index=False))
        print('\nConcluído.')

── Taxa de Mortalidade DCNT por 100 mil hab. ────────────────────

  [OK]  81,961 linhas  →  taxa_dcnt_municipio.parquet

Top 10 municípios por taxa DCNT/100k (exercício mais recente):
       nome_municipio sigla_estado  obitos_dcnt  populacao  taxa_dcnt_100k
        Pariquera-Açu           SP          471      19576         2406.01
 São João do Polêsine           RS           65       2707         2401.18
Campina Grande do Sul           PR         1041      49971         2083.21
             Barretos           SP         2207     126600         1743.29
             Aiuruoca           MG          105       6382         1645.25
       Augustinópolis           TO          265      18128         1461.83
        Dois Lajeados           RS           46       3158         1456.62
              Alecrim           RS           90       6228         1445.09
               Muriaé           MG         1495     108161         1382.20
          Campo Largo           PR         1956     142695       